# Image Classification: Facial Expression Recognition with CNN

**Part II - Vision Tasks | 6CS012 Final Portfolio Project**

This notebook implements an end-to-end deep learning pipeline for image classification using Convolutional Neural Networks (CNNs). The task is to classify facial expressions into 7 emotion categories: angry, disgust, fear, happy, neutral, sad, and surprise.


## 1. Setup & Imports


In [1]:
# Suppress TensorFlow/CUDA warnings
import os
os.environ['TF_CPP_MIN_LOG_LEVEL']      = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS']     = '0'

# ── Parallel GPU execution tunables ──────────────────────────────────────────
# Private thread per GPU for kernel submission — eliminates CPU-side bottleneck
os.environ['TF_GPU_THREAD_MODE']  = 'gpu_private'
os.environ['TF_GPU_THREAD_COUNT'] = '2'          # 2 threads / GPU (optimal for T4/P100)
# cuDNN frontend API — faster conv kernels on Ampere/Volta (T4) and Pascal (P100)
os.environ['TF_ENABLE_CUDNN_FRONTEND'] = '1'
# XLA auto-jit: level-2 fuses entire subgraphs (10-30% speedup on CNNs)
os.environ['TF_XLA_FLAGS'] = '--tf_xla_auto_jit=2 --tf_xla_cpu_global_jit'

import warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)

print('GPU thread mode  : gpu_private (2 threads/GPU)')
print('cuDNN frontend   : enabled')
print('XLA auto-jit     : level 2')

GPU thread mode  : gpu_private (2 threads/GPU)
cuDNN frontend   : enabled
XLA auto-jit     : level 2


In [2]:
# Set global random seeds for reproducibility
import random

SEED = 42
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

print(f"Random seed set to {SEED} for reproducibility.")

Random seed set to 42 for reproducibility.


In [3]:
# Core libraries
import numpy as np
import pandas as pd

np.random.seed(42)
print("Core libraries loaded. NumPy seed set.")

Core libraries loaded. NumPy seed set.


In [4]:
# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
print("Visualization libraries loaded.")

Visualization libraries loaded.


In [5]:
# Scikit-learn
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score
print("Scikit-learn loaded.")

Scikit-learn loaded.


In [6]:
# Install required packages (if not pre-installed)
!pip install pillow -q
print("Packages ready.")

Packages ready.


In [7]:
# TensorFlow / Keras
import tensorflow as tf
tf.random.set_seed(42)

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten, Dense,
                                      Dropout, BatchNormalization, Input,
                                      GlobalAveragePooling2D)
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mn_preprocess_input
from tensorflow.keras.regularizers import l2

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.21.0


In [8]:
import os

print('=== Hardware Diagnostics ===')
print(f'TensorFlow version: {tf.__version__}')
print(f'Physical devices  : {tf.config.list_physical_devices()}')
print()

DEVICE   = 'CPU'
strategy = tf.distribute.get_strategy()

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    strategy = tf.distribute.MirroredStrategy()
    DEVICE   = 'GPU'
    print(f'GPUs detected     : {len(gpus)}')

if DEVICE == 'CPU':
    try:
        tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
        tf.config.experimental_connect_to_cluster(tpu)
        tf.tpu.experimental.initialize_tpu_system(tpu)
        strategy = tf.distribute.TPUStrategy(tpu)
        DEVICE   = 'TPU'
        print(f'TPU cores         : {strategy.num_replicas_in_sync}')
    except Exception:
        pass

if DEVICE == 'CPU':
    print('No accelerator detected.')
    print('Kaggle: Settings → Accelerator → GPU T4 x2 or P100')

# ── XLA JIT compilation ───────────────────────────────────────────────────────
# Fuses GPU ops into single kernels; 10-30% speedup on CNN workloads.
tf.config.optimizer.set_jit(True)

# ── Mixed precision ───────────────────────────────────────────────────────────
if DEVICE in ('GPU', 'TPU'):
    tf.keras.mixed_precision.set_global_policy('mixed_float16')

# ── Thread parallelism ────────────────────────────────────────────────────────
# 0 = TF chooses optimal count automatically
tf.config.threading.set_inter_op_parallelism_threads(0)
tf.config.threading.set_intra_op_parallelism_threads(0)

n_gpus = strategy.num_replicas_in_sync
print(f'\nStrategy          : {strategy.__class__.__name__}')
print(f'Replicas (GPUs)   : {n_gpus}')
print(f'Mixed precision   : {tf.keras.mixed_precision.global_policy().name}')
print(f'XLA JIT           : enabled')
print(f'Device            : {DEVICE}')

=== Hardware Diagnostics ===
TensorFlow version: 2.21.0
Physical devices  : [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]

No accelerator detected.
Kaggle: Settings → Accelerator → GPU T4 x2 or P100

Strategy          : _DefaultDistributionStrategy
Replicas (GPUs)   : 1
Mixed precision   : float32
XLA JIT           : enabled
Device            : CPU


In [9]:
# Image processing
from PIL import Image
from collections import Counter
print("Image processing libraries loaded.")

Image processing libraries loaded.


## 2. Data Understanding, Analysis, Visualization & Cleaning


### 2.1 Define Dataset Paths


In [10]:
# Download / locate the FER dataset
import os

dataset_path = None

# ── 1. Check common Kaggle input paths (dataset added via UI) ─────────────────
_candidates = [
    '/kaggle/input/facial-expression-classification',
    '/kaggle/input/facial-expression-classification/facial expression classification',
    '/kaggle/input/datasets/kamaldhitalofficial/facial-expression-classification/facial expression classification',
]
for _p in _candidates:
    if os.path.isdir(_p):
        # Prefer the sub-folder that contains train/validation directly
        if os.path.isdir(os.path.join(_p, 'train')):
            dataset_path = _p
            break
        # Otherwise look one level deeper
        for _item in os.listdir(_p):
            _sub = os.path.join(_p, _item)
            if os.path.isdir(os.path.join(_sub, 'train')):
                dataset_path = _sub
                break
    if dataset_path:
        break

# ── 2. Fall back: download via Kaggle API ─────────────────────────────────────
if dataset_path is None:
    _work = '/kaggle/working'
    os.makedirs(_work, exist_ok=True)          # ensure the directory exists first

    print("Dataset not found in /kaggle/input — downloading via Kaggle API...")
    os.environ.setdefault('KAGGLE_USERNAME', 'kamaldhital')
    os.environ.setdefault('KAGGLE_KEY',      'c638db8e0004e4f1be783e66b23bf023')
    os.system('pip install kaggle -q')
    ret = os.system(
        f'kaggle datasets download -d kamaldhital/facial-expression-classification '
        f'-p {_work} --unzip -q'
    )
    if ret != 0:
        raise RuntimeError("Kaggle download failed. Check KAGGLE_USERNAME / KAGGLE_KEY.")

    # Walk to find the folder that has train/ and validation/ sub-dirs
    for root, dirs, _ in os.walk(_work):
        if 'train' in dirs and 'validation' in dirs:
            dataset_path = root
            break

# ── 3. Final guard ────────────────────────────────────────────────────────────
if dataset_path is None or not os.path.isdir(dataset_path):
    raise FileNotFoundError(
        "Could not locate dataset with train/ and validation/ sub-folders.\n"
        "In Kaggle: click 'Add Data' → search 'facial-expression-classification'."
    )

print(f"Dataset path : {dataset_path}")
print(f"Contents     : {sorted(os.listdir(dataset_path))}")

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata
Downloaded. Dataset at: /kaggle/working


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working'

In [ ]:
# Define dataset paths
import shutil

if dataset_path is None:
    raise FileNotFoundError("Dataset not found! Please add the dataset to your Kaggle notebook via 'Add Data' button.")

# On Kaggle, /kaggle/input is read-only so we copy to /kaggle/working for modification
if '/kaggle/input' in dataset_path:
    DATASET_DIR = '/kaggle/working/facial expression classification'
    if not os.path.exists(DATASET_DIR):
        print("Copying dataset to writable directory...")
        shutil.copytree(dataset_path, DATASET_DIR)
        print("Done!")
    else:
        print("Dataset already copied.")
else:
    DATASET_DIR = dataset_path

TRAIN_DIR = os.path.join(DATASET_DIR, 'train')
VAL_DIR = os.path.join(DATASET_DIR, 'validation')
TEST_DIR = os.path.join(DATASET_DIR, 'test')

classes = sorted(os.listdir(TRAIN_DIR))
NUM_CLASSES = len(classes)
print(f"Dataset path: {DATASET_DIR}")
print(f"Number of classes: {NUM_CLASSES}")
print(f"Classes: {classes}")


### 2.2 Count Total Images


In [ ]:
# Count images per class in each split
def count_images(directory):
    counts = {}
    for cls in sorted(os.listdir(directory)):
        cls_path = os.path.join(directory, cls)
        if os.path.isdir(cls_path):
            counts[cls] = len(os.listdir(cls_path))
    return counts

train_counts = count_images(TRAIN_DIR)
val_counts = count_images(VAL_DIR)
test_counts = count_images(TEST_DIR)

print("Training set:")
for cls, count in train_counts.items():
    print(f"  {cls:<10}: {count}")
print(f"  {'TOTAL':<10}: {sum(train_counts.values())}")

In [ ]:
# Validation set counts
print("Validation set:")
for cls, count in val_counts.items():
    print(f"  {cls:<10}: {count}")
print(f"  {'TOTAL':<10}: {sum(val_counts.values())}")

print(f"\nTest set:")
for cls, count in test_counts.items():
    print(f"  {cls:<10}: {count}")
print(f"  {'TOTAL':<10}: {sum(test_counts.values())}")

In [ ]:
# Grand total
total_train = sum(train_counts.values())
total_val = sum(val_counts.values())
total_test = sum(test_counts.values())
grand_total = total_train + total_val + total_test

print(f"Grand Total Images: {grand_total}")
print(f"  Train:      {total_train} ({total_train/grand_total*100:.1f}%)")
print(f"  Validation: {total_val} ({total_val/grand_total*100:.1f}%)")
print(f"  Test:       {total_test} ({total_test/grand_total*100:.1f}%)")

In [ ]:
# Summary table of all splits
summary_data = {
    'Class': classes,
    'Train': [train_counts[c] for c in classes],
    'Validation': [val_counts[c] for c in classes],
    'Test': [test_counts[c] for c in classes],
}
summary_df = pd.DataFrame(summary_data)
summary_df['Total'] = summary_df['Train'] + summary_df['Validation'] + summary_df['Test']
summary_df.loc[len(summary_df)] = ['TOTAL', sum(train_counts.values()), 
                                     sum(val_counts.values()), sum(test_counts.values()),
                                     grand_total]
summary_df

### 2.3 Class Distribution Visualization


In [ ]:
# Bar chart - Training set class distribution
fig, ax = plt.subplots(figsize=(10, 5))

colors = ['#e74c3c', '#9b59b6', '#3498db', '#2ecc71', '#95a5a6', '#e67e22', '#f1c40f']
bars = ax.bar(classes, [train_counts[c] for c in classes], color=colors, edgecolor='black')

# Add count labels on top of bars
for bar, cls in zip(bars, classes):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 50, 
            str(int(height)), ha='center', va='bottom', fontweight='bold')

ax.set_xlabel('Emotion Class', fontsize=12)
ax.set_ylabel('Number of Images', fontsize=12)
ax.set_title('Training Set - Class Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Analyze class imbalance
max_class = max(train_counts, key=train_counts.get)
min_class = min(train_counts, key=train_counts.get)
imbalance_ratio = train_counts[max_class] / train_counts[min_class]

print(f"Most represented class:  '{max_class}' with {train_counts[max_class]} images")
print(f"Least represented class: '{min_class}' with {train_counts[min_class]} images")
print(f"Imbalance ratio: {imbalance_ratio:.1f}x")
print(f"\nNote: Significant class imbalance detected. 'disgust' is severely underrepresented.")
print("We will use class weights during training to handle this.")

In [ ]:
# Grouped bar chart - All splits comparison
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(classes))
width = 0.25

bars1 = ax.bar(x - width, [train_counts[c] for c in classes], width, label='Train', color='#3498db')
bars2 = ax.bar(x, [val_counts[c] for c in classes], width, label='Validation', color='#2ecc71')
bars3 = ax.bar(x + width, [test_counts[c] for c in classes], width, label='Test', color='#e74c3c')

ax.set_xlabel('Emotion Class', fontsize=12)
ax.set_ylabel('Number of Images', fontsize=12)
ax.set_title('Class Distribution Across All Splits', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(classes)
ax.legend()
plt.tight_layout()
plt.show()

### 2.4 Inspect Image Properties


In [ ]:
# Check image properties (size, mode, format)
sample_path = os.path.join(TRAIN_DIR, 'happy', os.listdir(os.path.join(TRAIN_DIR, 'happy'))[0])
sample_img = Image.open(sample_path)

print(f"Sample image path: {sample_path}")
print(f"Image size: {sample_img.size}")
print(f"Image mode: {sample_img.mode} ({'Grayscale' if sample_img.mode == 'L' else 'RGB'})")
print(f"Image format: {sample_img.format}")

In [ ]:
# Verify image integrity and remove corrupted files
import random as rng

corrupted_files = []

for split_name, split_dir in [('train', TRAIN_DIR), ('validation', VAL_DIR), ('test', TEST_DIR)]:
    for cls in classes:
        cls_dir = os.path.join(split_dir, cls)
        files = os.listdir(cls_dir)
        for f in files:
            fpath = os.path.join(cls_dir, f)
            try:
                img = Image.open(fpath)
                img.verify()
            except Exception:
                corrupted_files.append(fpath)

if corrupted_files:
    print(f"Found {len(corrupted_files)} corrupted/unreadable images:")
    for path in corrupted_files:
        print(f"  {path}")
    
    # Remove corrupted files
    for path in corrupted_files:
        os.remove(path)
    print(f"\nRemoved {len(corrupted_files)} corrupted files.")
else:
    print("No corrupted images found. All files are valid.")

print("All valid images are 48x48 pixels. Dataset is consistent.")

### 2.4.1 Detect Duplicate Images Across Splits


In [ ]:
# Compute file hashes to detect duplicate images across splits
import hashlib

def compute_image_hashes(directory):
    """Compute MD5 hash for each image file in the directory."""
    hashes = {}
    for cls in sorted(os.listdir(directory)):
        cls_dir = os.path.join(directory, cls)
        if not os.path.isdir(cls_dir):
            continue
        for f in os.listdir(cls_dir):
            fpath = os.path.join(cls_dir, f)
            try:
                with open(fpath, 'rb') as img_file:
                    file_hash = hashlib.md5(img_file.read()).hexdigest()
                hashes[fpath] = file_hash
            except Exception:
                continue
    return hashes

print("Computing image hashes for all splits...")
train_hashes = compute_image_hashes(TRAIN_DIR)
val_hashes = compute_image_hashes(VAL_DIR)
test_hashes = compute_image_hashes(TEST_DIR)

print(f"  Train images hashed: {len(train_hashes)}")
print(f"  Validation images hashed: {len(val_hashes)}")
print(f"  Test images hashed: {len(test_hashes)}")

In [ ]:
# Find duplicate images across splits
train_hash_set = set(train_hashes.values())
val_hash_set = set(val_hashes.values())
test_hash_set = set(test_hashes.values())

# Check overlaps between splits
train_val_overlap = train_hash_set & val_hash_set
train_test_overlap = train_hash_set & test_hash_set
val_test_overlap = val_hash_set & test_hash_set

print("Cross-split duplicate analysis:")
print(f"  Train ∩ Validation: {len(train_val_overlap)} duplicate images")
print(f"  Train ∩ Test:       {len(train_test_overlap)} duplicate images")
print(f"  Validation ∩ Test:  {len(val_test_overlap)} duplicate images")

total_cross_dups = len(train_val_overlap | train_test_overlap | val_test_overlap)
if total_cross_dups > 0:
    print(f"\nTotal unique duplicates found across splits: {total_cross_dups}")
    print("These must be removed to prevent data leakage!")
else:
    print("\nNo cross-split duplicates found. No data leakage risk.")

In [ ]:
# Check for duplicates within each split
def find_within_duplicates(hashes, split_name):
    hash_to_files = {}
    for fpath, h in hashes.items():
        hash_to_files.setdefault(h, []).append(fpath)
    duplicates = {h: files for h, files in hash_to_files.items() if len(files) > 1}
    dup_count = sum(len(files) - 1 for files in duplicates.values())
    print(f"  {split_name}: {dup_count} duplicate images ({len(duplicates)} unique images repeated)")
    return duplicates

print("Within-split duplicate analysis:")
train_within_dups = find_within_duplicates(train_hashes, "Train")
val_within_dups = find_within_duplicates(val_hashes, "Validation")
test_within_dups = find_within_duplicates(test_hashes, "Test")

In [ ]:
# Remove cross-split duplicates (keep in train, remove from val/test)
# Remove within-split duplicates (keep first occurrence)
removed_count = 0

# 1. Remove from validation if duplicate exists in train
for fpath, h in list(val_hashes.items()):
    if h in train_hash_set:
        if os.path.exists(fpath):
            os.remove(fpath)
            removed_count += 1

# 2. Remove from test if duplicate exists in train or validation
for fpath, h in list(test_hashes.items()):
    if h in train_hash_set or h in val_hash_set:
        if os.path.exists(fpath):
            os.remove(fpath)
            removed_count += 1

# 3. Remove within-split duplicates (keep first, remove rest)
for split_dups in [train_within_dups, val_within_dups, test_within_dups]:
    for h, files in split_dups.items():
        for fpath in files[1:]:  # Keep first, remove rest
            if os.path.exists(fpath):
                os.remove(fpath)
                removed_count += 1

print(f"Total duplicate images removed: {removed_count}")

In [ ]:
# Verify dataset after duplicate removal
print("Dataset after cleaning:")
train_counts_clean = count_images(TRAIN_DIR)
val_counts_clean = count_images(VAL_DIR)
test_counts_clean = count_images(TEST_DIR)

total_train = sum(train_counts_clean.values())
total_val = sum(val_counts_clean.values())
total_test = sum(test_counts_clean.values())
grand_total = total_train + total_val + total_test

print(f"  Train:      {total_train}")
print(f"  Validation: {total_val}")
print(f"  Test:       {total_test}")
print(f"  Grand Total: {grand_total}")

# Update counts for downstream use
train_counts = train_counts_clean
val_counts = val_counts_clean
test_counts = test_counts_clean

print("\nCounts updated. Dataset is clean and ready.")

In [ ]:
# Display sample images from each class
fig, axes = plt.subplots(2, 7, figsize=(16, 5))

for idx, cls in enumerate(classes):
    cls_dir = os.path.join(TRAIN_DIR, cls)
    files = os.listdir(cls_dir)
    
    # Find valid images (skip corrupted ones)
    valid_imgs = []
    for f in files:
        try:
            img = Image.open(os.path.join(cls_dir, f))
            img.load()
            valid_imgs.append(img)
            if len(valid_imgs) == 2:
                break
        except Exception:
            continue
    
    # First row - sample 1
    axes[0, idx].imshow(valid_imgs[0], cmap='gray')
    axes[0, idx].set_title(cls, fontsize=10, fontweight='bold')
    axes[0, idx].axis('off')
    
    # Second row - sample 2
    axes[1, idx].imshow(valid_imgs[1], cmap='gray')
    axes[1, idx].axis('off')

fig.suptitle('Sample Images from Each Emotion Class', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 2.6 Pixel Intensity Distribution


In [ ]:
# Pixel intensity distribution for each class
fig, axes = plt.subplots(1, 7, figsize=(18, 3))

for idx, cls in enumerate(classes):
    cls_dir = os.path.join(TRAIN_DIR, cls)
    files = os.listdir(cls_dir)[:150]  # Sample images per class
    
    all_pixels = []
    for f in files:
        try:
            img = np.array(Image.open(os.path.join(cls_dir, f)))
            all_pixels.extend(img.flatten())
        except Exception:
            continue  # Skip corrupted images
    
    axes[idx].hist(all_pixels, bins=50, color=colors[idx], alpha=0.7, density=True)
    axes[idx].set_title(cls, fontsize=9, fontweight='bold')
    axes[idx].set_xlim(0, 255)
    axes[idx].set_ylim(0, 0.025)
    if idx == 0:
        axes[idx].set_ylabel('Density')

fig.suptitle('Pixel Intensity Distribution per Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.7 Dataset Split Rationale


In [ ]:
# Pie chart - Dataset split proportions
fig, ax = plt.subplots(figsize=(6, 6))

sizes = [total_train, total_val, total_test]
labels_pie = [f'Train\n{total_train} ({total_train/grand_total*100:.1f}%)', 
              f'Validation\n{total_val} ({total_val/grand_total*100:.1f}%)', 
              f'Test\n{total_test} ({total_test/grand_total*100:.1f}%)']
colors_pie = ['#3498db', '#2ecc71', '#e74c3c']

ax.pie(sizes, labels=labels_pie, colors=colors_pie, autopct='', startangle=90,
       textprops={'fontsize': 11}, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax.set_title('Dataset Split Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Split Rationale:")
print(f"  - Training ({total_train/grand_total*100:.1f}%): Largest portion for model learning")
print(f"  - Validation ({total_val/grand_total*100:.1f}%): Used for hyperparameter tuning and early stopping")
print(f"  - Test ({total_test/grand_total*100:.1f}%): Held-out set for final unbiased evaluation")

### 2.8 Preprocessing & Data Generators


The following preprocessing steps are applied:
- **Rescaling**: Pixel values normalized from [0, 255] to [0, 1]
- **Target Size**: Images resized to 48x48 (already native size)
- **Color Mode**: Grayscale (single channel)
- **Batch Size**: 64 for efficient GPU training


In [ ]:
# Preprocessing constants
IMG_SIZE    = (48, 48)
COLOR_MODE  = 'grayscale'
INPUT_SHAPE = (48, 48, 1)
AUTOTUNE    = tf.data.AUTOTUNE

# Scale batch size with GPU count — each GPU gets 512 samples.
# T4×2 → 1024 | P100 → 512
BATCH_SIZE = 512 * strategy.num_replicas_in_sync
print(f'Image size  : {IMG_SIZE}')
print(f'Batch size  : {BATCH_SIZE}  ({512} per GPU × {strategy.num_replicas_in_sync} GPU(s))')
print(f'Input shape : {INPUT_SHAPE}')

In [ ]:
# ── tf.data performance options ───────────────────────────────────────────────
# AutoShardPolicy.DATA: MirroredStrategy shards by rows (not files) across GPUs.
# private_threadpool_size: all CPU threads available for parallel decoding/augmentation.
_ds_options = tf.data.Options()
_ds_options.experimental_distribute.auto_shard_policy = (
    tf.data.experimental.AutoShardPolicy.DATA
)
_ds_options.threading.private_threadpool_size = 48   # use all Kaggle vCPUs
_ds_options.threading.max_intra_op_parallelism = 1   # prevents thread oversubscription

# ── GPU-accelerated augmentation pipeline ─────────────────────────────────────
aug_pipeline = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(10 / 360),
    layers.RandomZoom(0.10),
    layers.RandomTranslation(0.10, 0.10),
    layers.RandomContrast(0.05),
], name='aug_pipeline')

def augment(image, label):
    return tf.clip_by_value(aug_pipeline(image[tf.newaxis], training=True)[0], 0.0, 1.0), label

def make_dataset(directory, augment_data=False):
    ds = tf.keras.utils.image_dataset_from_directory(
        str(directory),
        labels='inferred',
        label_mode='categorical',
        color_mode='grayscale',
        batch_size=None,
        image_size=IMG_SIZE,
        shuffle=False,          # shuffle handled explicitly below for better control
        seed=42,
        class_names=classes,
    )
    # Normalise on CPU in parallel, then cache in memory
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y),
                num_parallel_calls=AUTOTUNE)
    ds = ds.cache()             # RAM cache: eliminates all disk I/O after epoch 1

    if augment_data:
        # Shuffle AFTER cache so every epoch sees a different order
        ds = ds.shuffle(buffer_size=27000, seed=42, reshuffle_each_iteration=True)
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)

    ds = ds.batch(BATCH_SIZE, drop_remainder=True)  # drop_remainder keeps shapes static → XLA-friendly
    ds = ds.prefetch(AUTOTUNE)  # overlap CPU pre-processing with GPU compute
    ds = ds.with_options(_ds_options)
    return ds

train_ds = make_dataset(TRAIN_DIR, augment_data=True)
val_ds   = make_dataset(VAL_DIR)
test_ds  = make_dataset(TEST_DIR)

# Precompute true labels once — avoids test_generator.reset() + .classes in every eval cell
true_classes = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in test_ds])

print(f'Train batches : {len(train_ds)} | Val : {len(val_ds)} | Test : {len(test_ds)}')
print(f'Batch size    : {BATCH_SIZE}')
print(f'True classes  : {true_classes.shape}  unique={sorted(set(true_classes.tolist()))}')

for bx, by in train_ds.take(1):
    print(f'Batch shape   : images={bx.shape}, labels={by.shape}')
    print(f'Pixel range   : [{bx.numpy().min():.3f}, {bx.numpy().max():.3f}]')
    print(f'Dtype         : {bx.dtype}')

In [ ]:
# Validation data generator - rescaling only
val_datagen = ImageDataGenerator(rescale=1./255)

val_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode=COLOR_MODE,
    class_mode='categorical',
    shuffle=False,
    seed=SEED
)

In [ ]:
# Test data generator - rescaling only
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode=COLOR_MODE,
    class_mode='categorical',
    shuffle=False,
    seed=SEED
)

### 2.9 Data Augmentation


Data augmentation helps:
- Increase effective training set size
- Reduce overfitting
- Handle class imbalance (especially for underrepresented classes like 'disgust')

The following augmentations are applied to training data:


In [ ]:
# Training data generator WITH augmentation (to reduce overfitting)
train_datagen_aug = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
    fill_mode='nearest'
)

train_generator_aug = train_datagen_aug.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode=COLOR_MODE,
    class_mode='categorical',
    shuffle=True,
    seed=SEED
)

# Verify data is loading correctly
print(f"\nClass indices: {train_generator_aug.class_indices}")
print(f"Number of samples: {train_generator_aug.samples}")
print(f"Batches per epoch: {len(train_generator_aug)}")

# Quick sanity check
batch_x, batch_y = next(train_generator_aug)
print(f"\nBatch shape: images={batch_x.shape}, labels={batch_y.shape}")
print(f"Pixel range: min={batch_x.min():.4f}, max={batch_x.max():.4f}")


### 2.10 Visualize Augmented vs Original Images


In [ ]:
# Show original vs augmented images
sample_cls = 'happy'
sample_dir = os.path.join(TRAIN_DIR, sample_cls)

# Find a valid image
original_img = None
for f in os.listdir(sample_dir):
    try:
        original_img = Image.open(os.path.join(sample_dir, f))
        original_img.load()
        break
    except Exception:
        continue

# Keep raw pixel values (0-255) for augmentation
original_arr = np.array(original_img).reshape(1, 48, 48, 1).astype('float32')

# Augmentation generator with rescale (normalizes after augmenting)
aug_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    zoom_range=0.15,
    shear_range=0.15,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))

# Row 1: Original (repeated)
for i in range(5):
    axes[0, i].imshow(original_arr[0].reshape(48, 48), cmap='gray', vmin=0, vmax=255)
    axes[0, i].set_title('Original' if i == 2 else '', fontsize=10)
    axes[0, i].axis('off')

# Row 2: Augmented versions
aug_iter = aug_gen.flow(original_arr, batch_size=1, seed=None)
for i in range(5):
    aug_img = next(aug_iter)[0].reshape(48, 48)
    axes[1, i].imshow(aug_img, cmap='gray', vmin=0, vmax=1)
    axes[1, i].set_title(f'Augmented {i+1}', fontsize=10)
    axes[1, i].axis('off')

fig.suptitle(f'Original vs Augmented Images (class: {sample_cls})', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Show augmented samples across all classes
fig, axes = plt.subplots(7, 6, figsize=(14, 16))

aug_vis_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    zoom_range=0.15,
    shear_range=0.15,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

for row, cls in enumerate(classes):
    cls_dir = os.path.join(TRAIN_DIR, cls)
    
    # Find a valid image (keep raw 0-255 values)
    img_arr = None
    for f in os.listdir(cls_dir):
        try:
            pil_img = Image.open(os.path.join(cls_dir, f))
            pil_img.load()
            img_arr = np.array(pil_img).reshape(1, 48, 48, 1).astype('float32')
            break
        except Exception:
            continue
    
    # Column 0: Original
    axes[row, 0].imshow(img_arr[0].reshape(48, 48), cmap='gray', vmin=0, vmax=255)
    axes[row, 0].set_title('Original' if row == 0 else '', fontsize=9)
    axes[row, 0].set_ylabel(cls, fontsize=10, fontweight='bold', rotation=0, labelpad=50)
    axes[row, 0].axis('off')
    
    # Columns 1-5: Augmented (rescale inside generator normalizes to 0-1)
    aug_iter = aug_vis_gen.flow(img_arr, batch_size=1)
    for col in range(1, 6):
        aug_img = next(aug_iter)[0].reshape(48, 48)
        axes[row, col].imshow(aug_img, cmap='gray', vmin=0, vmax=1)
        axes[row, col].set_title(f'Aug {col}' if row == 0 else '', fontsize=9)
        axes[row, col].axis('off')

fig.suptitle('Augmented Samples Across All Emotion Classes', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


### 2.11 Compute Class Weights for Imbalance


In [ ]:
# Compute class weights from disk — independent of ImageDataGenerator
import pathlib
from sklearn.utils.class_weight import compute_class_weight

_all_labels = []
for i, cls in enumerate(classes):
    n = len(list(pathlib.Path(TRAIN_DIR).glob(f'{cls}/*.jpg')))
    _all_labels.extend([i] * n)

_cw_arr    = compute_class_weight('balanced',
                                   classes=np.unique(_all_labels),
                                   y=np.array(_all_labels))
class_weights = dict(enumerate(_cw_arr))

print('Class weights (inverse-frequency, handles imbalance):')
for idx, cls in enumerate(classes):
    print(f'  {cls:<10}: {class_weights[idx]:.4f}')
print('\nHigher weight → under-represented class gets proportionally more influence.')

### 2.12 Data Understanding Summary

**Dataset**: Facial Expression Classification
- **Classes**: 7 emotions (angry, disgust, fear, happy, neutral, sad, surprise)
- **Image Size**: 48×48 pixels, grayscale (1 channel)

**Data Cleaning Applied:**
1. Corrupted/unreadable images detected and removed
2. Cross-split duplicates removed (prevents data leakage)
3. Within-split duplicates removed

**Handling Class Imbalance:**
- Class weights computed (no oversampling — avoids overfitting from synthetic duplicates)

**Preventing Overfitting:**
- Stronger data augmentation (rotation ±20°, shift ±15%, brightness, zoom, shear)
- L2 regularization (1e-4) on all Conv and Dense layers
- Higher Dropout rates (0.3 conv, 0.6 dense)
- EarlyStopping + ReduceLROnPlateau

The dataset is now clean and ready for model building.

## 3. Design, Train, and Evaluate a Baseline Model

This section builds a baseline CNN from scratch with the following architecture:
- **3 Convolutional layers**, each followed by MaxPooling
- **3 Fully Connected (Dense) layers**
- **Output layer** with softmax activation for 7-class classification

### 3.1 Model Architecture

In [ ]:
# Build baseline CNN model (3 Conv + 3 FC as required)

def build_baseline_cnn(input_shape=(48, 48, 1), num_classes=7):
    inp = tf.keras.Input(shape=input_shape)
    
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                      kernel_initializer='he_normal')(inp)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same',
                      kernel_initializer='he_normal')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same',
                      kernel_initializer='he_normal')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='relu', kernel_initializer='he_normal')(x)
    x = layers.Dense(128, activation='relu', kernel_initializer='he_normal')(x)
    x = layers.Dense(64, activation='relu', kernel_initializer='he_normal')(x)
    
    out = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
    return tf.keras.Model(inp, out, name='baseline_cnn')

with strategy.scope():
    baseline_model = build_baseline_cnn(INPUT_SHAPE, NUM_CLASSES)
baseline_model.summary()


### 3.2 Model Summary

In [ ]:
# Print model summary
baseline_model.summary()

In [ ]:
# Visualize model architecture details
print("=" * 60)
print("BASELINE CNN ARCHITECTURE DETAILS")
print("=" * 60)
print(f"Input Shape:        {INPUT_SHAPE}")
print(f"Number of Classes:  {NUM_CLASSES}")
print()
print("Layer Details:")
print("-" * 60)
print(f"Conv2D Layer 1:     32 filters, 3x3 kernel, ReLU, same padding")
print(f"MaxPooling2D 1:     2x2 pool size")
print(f"Conv2D Layer 2:     64 filters, 3x3 kernel, ReLU, same padding")
print(f"MaxPooling2D 2:     2x2 pool size")
print(f"Conv2D Layer 3:     128 filters, 3x3 kernel, ReLU, same padding")
print(f"MaxPooling2D 3:     2x2 pool size")
print(f"Dense Layer 1:      256 neurons, ReLU")
print(f"Dense Layer 2:      128 neurons, ReLU")
print(f"Dense Layer 3:      64 neurons, ReLU")
print(f"Output Layer:       {NUM_CLASSES} neurons, Softmax")
print("-" * 60)
print(f"Total Parameters:   {baseline_model.count_params():,}")
trainable = sum([tf.keras.backend.count_params(w) for w in baseline_model.trainable_weights])
print(f"Trainable Params:   {trainable:,}")

### 3.3 Compile the Model

In [ ]:
# Compile baseline model
with strategy.scope():
    baseline_model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
print("Baseline model compiled with Adam optimizer.")


### 3.4 Train the Baseline Model


In [ ]:
# Train baseline model
import time

EPOCHS = 100

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

print(f"Training baseline model for up to {EPOCHS} epochs...")
print(f"Using device: {DEVICE} | Batch size: {BATCH_SIZE}")
print()

start_time = time.time()

baseline_history = baseline_model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    class_weight=class_weights,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

baseline_train_time = time.time() - start_time
print(f"\nTraining completed in {baseline_train_time:.1f} seconds ({baseline_train_time/60:.1f} minutes)")


### 3.5 Training & Validation Curves


In [ ]:
# Plot training vs validation accuracy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(baseline_history.history['accuracy']) + 1)

# Accuracy
axes[0].plot(epochs_range, baseline_history.history['accuracy'], 'b-', label='Training Accuracy')
axes[0].plot(epochs_range, baseline_history.history['val_accuracy'], 'r-', label='Validation Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Baseline CNN - Training vs Validation Accuracy', fontweight='bold')
axes[0].legend()
axes[0].grid(True)

# Loss
axes[1].plot(epochs_range, baseline_history.history['loss'], 'b-', label='Training Loss')
axes[1].plot(epochs_range, baseline_history.history['val_loss'], 'r-', label='Validation Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Baseline CNN - Training vs Validation Loss', fontweight='bold')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Print final training metrics
final_train_acc = baseline_history.history['accuracy'][-1]
final_val_acc = baseline_history.history['val_accuracy'][-1]
final_train_loss = baseline_history.history['loss'][-1]
final_val_loss = baseline_history.history['val_loss'][-1]
best_val_acc = max(baseline_history.history['val_accuracy'])
best_epoch = baseline_history.history['val_accuracy'].index(best_val_acc) + 1

print("Baseline Model - Final Training Metrics:")
print(f"  Final Training Accuracy:     {final_train_acc:.4f}")
print(f"  Final Validation Accuracy:   {final_val_acc:.4f}")
print(f"  Final Training Loss:         {final_train_loss:.4f}")
print(f"  Final Validation Loss:       {final_val_loss:.4f}")
print(f"  Best Validation Accuracy:    {best_val_acc:.4f} (Epoch {best_epoch})")
print(f"  Total Epochs Trained:        {len(baseline_history.history['accuracy'])}")
print(f"  Training Time:               {baseline_train_time:.1f}s")

### 3.6 Model Evaluation on Test Set


In [ ]:
# Evaluate on test set
print('Evaluating baseline model on test set...')
baseline_test_loss, baseline_test_acc = baseline_model.evaluate(test_ds, verbose=1)

print(f'\nBaseline Model - Test Results:')
print(f'  Test Accuracy: {baseline_test_acc:.4f}')
print(f'  Test Loss:     {baseline_test_loss:.4f}')

In [ ]:
# Generate predictions on test set (test_ds is deterministic — no reset needed)
baseline_predictions  = baseline_model.predict(test_ds, verbose=1)
baseline_pred_classes = np.argmax(baseline_predictions, axis=1)
# true_classes was precomputed in cell 41 from test_ds

print(f'Predictions generated: {len(baseline_pred_classes)} samples')

In [ ]:
# Classification report
print("Baseline CNN - Classification Report:")
print("=" * 60)
print(classification_report(true_classes, baseline_pred_classes, target_names=classes))

In [ ]:
# Confusion matrix
cm = confusion_matrix(true_classes, baseline_pred_classes)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes, ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title('Baseline CNN - Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Per-class accuracy
print("Baseline CNN - Per-Class Accuracy:")
print("-" * 40)
for i, cls in enumerate(classes):
    cls_mask = true_classes == i
    cls_acc = np.mean(baseline_pred_classes[cls_mask] == i)
    print(f"  {cls:<10}: {cls_acc:.4f} ({np.sum(cls_mask)} samples)")

### 3.7 Inference on Sample Images


In [ ]:
# Inference on sample test images
fig, axes = plt.subplots(2, 5, figsize=(16, 7))

test_batch, test_labels = next(iter(test_ds))
test_batch  = test_batch.numpy()
test_labels = test_labels.numpy()

for i in range(10):
    row, col   = i // 5, i % 5
    img        = test_batch[i, :, :, 0]           # squeeze channel dim
    true_label = classes[np.argmax(test_labels[i])]

    pred_probs  = baseline_model.predict(test_batch[i:i+1], verbose=0)
    pred_label  = classes[np.argmax(pred_probs)]
    confidence  = np.max(pred_probs) * 100

    axes[row, col].imshow(img, cmap='gray')
    color = 'green' if pred_label == true_label else 'red'
    axes[row, col].set_title(f'True: {true_label}\nPred: {pred_label} ({confidence:.1f}%)',
                              fontsize=9, color=color, fontweight='bold')
    axes[row, col].axis('off')

fig.suptitle('Baseline CNN - Sample Predictions (Green=Correct, Red=Wrong)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.8 Baseline Model Observations


## 4. Deeper Architecture with Regularization

This section extends the baseline model by:
- **Doubling the convolutional layers** (6 Conv layers instead of 3)
- **Increasing filter counts** (64→128→256 with two Conv layers per block)
- Adding **Batch Normalization** after each Conv layer
- Adding **Dropout** to prevent overfitting

### 4.1 Model Architecture

In [ ]:
# Build deeper CNN with regularization
from tensorflow.keras import regularizers

def build_deeper_cnn(input_shape=(48, 48, 1), num_classes=7):
    reg = regularizers.l2(1e-4)
    inp = tf.keras.Input(shape=input_shape)
    
    # Block 1
    x = layers.Conv2D(64, (3, 3), padding='same', kernel_regularizer=reg,
                      activation='relu', kernel_initializer='he_normal')(inp)
    x = layers.Conv2D(64, (3, 3), padding='same', kernel_regularizer=reg,
                      activation='relu', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    # Block 2
    x = layers.Conv2D(128, (3, 3), padding='same', kernel_regularizer=reg,
                      activation='relu', kernel_initializer='he_normal')(x)
    x = layers.Conv2D(128, (3, 3), padding='same', kernel_regularizer=reg,
                      activation='relu', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    # Block 3
    x = layers.Conv2D(256, (3, 3), padding='same', kernel_regularizer=reg,
                      activation='relu', kernel_initializer='he_normal')(x)
    x = layers.Conv2D(256, (3, 3), padding='same', kernel_regularizer=reg,
                      activation='relu', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Flatten()(x)
    
    x = layers.Dense(512, kernel_regularizer=reg, activation='relu',
                     kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    
    x = layers.Dense(256, kernel_regularizer=reg, activation='relu',
                     kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    
    x = layers.Dense(128, kernel_regularizer=reg, activation='relu',
                     kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    
    out = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
    return tf.keras.Model(inp, out, name='deeper_cnn')

with strategy.scope():
    deeper_model = build_deeper_cnn(INPUT_SHAPE, NUM_CLASSES)
deeper_model.summary()


### 4.2 Model Summary

In [ ]:
# Print deeper model summary
deeper_model.summary()

In [ ]:
# Compare architecture: Baseline vs Deeper
print("=" * 65)
print("ARCHITECTURE COMPARISON: BASELINE vs DEEPER CNN")
print("=" * 65)
print(f"{'Feature':<30} {'Baseline':<15} {'Deeper':<15}")
print("-" * 65)
print(f"{'Conv Layers':<30} {'3':<15} {'6':<15}")
print(f"{'Filters':<30} {'32→64→128':<15} {'64→128→256':<15}")
print(f"{'BatchNormalization':<30} {'No':<15} {'Yes (9 layers)':<15}")
print(f"{'Dropout':<30} {'No':<15} {'Yes (0.25/0.5)':<15}")
print(f"{'FC Layers':<30} {'3':<15} {'3':<15}")
print(f"{'FC Neurons':<30} {'256→128→64':<15} {'512→256→128':<15}")
print(f"{'Total Parameters':<30} {baseline_model.count_params():,}{'':>5} {deeper_model.count_params():,}")
print("-" * 65)

### 4.3 Compile the Model

In [ ]:
# Compile deeper model
with strategy.scope():
    deeper_model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
print("Deeper model compiled.")
print(f"  Optimizer: Adam (default lr=0.001)")
print(f"  Loss: categorical_crossentropy")


### 4.4 Train the Deeper Model

In [ ]:
# Train deeper model
EPOCHS_DEEP = 100

early_stopping_deep = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr_deep = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

print(f"Training deeper model for up to {EPOCHS_DEEP} epochs...")
print(f"Using device: {DEVICE} | Batch size: {BATCH_SIZE}")
print()

start_time_deep = time.time()

deeper_history = deeper_model.fit(
    train_ds,
    epochs=EPOCHS_DEEP,
    validation_data=val_ds,
    class_weight=class_weights,
    callbacks=[early_stopping_deep, reduce_lr_deep],
    verbose=1
)

deeper_train_time = time.time() - start_time_deep
print(f"\nTraining completed in {deeper_train_time:.1f} seconds ({deeper_train_time/60:.1f} minutes)")


### 4.5 Training & Validation Curves

In [ ]:
# Plot deeper model training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range_deep = range(1, len(deeper_history.history['accuracy']) + 1)

# Accuracy
axes[0].plot(epochs_range_deep, deeper_history.history['accuracy'], 'b-', label='Training Accuracy')
axes[0].plot(epochs_range_deep, deeper_history.history['val_accuracy'], 'r-', label='Validation Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Deeper CNN - Training vs Validation Accuracy', fontweight='bold')
axes[0].legend()
axes[0].grid(True)

# Loss
axes[1].plot(epochs_range_deep, deeper_history.history['loss'], 'b-', label='Training Loss')
axes[1].plot(epochs_range_deep, deeper_history.history['val_loss'], 'r-', label='Validation Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Deeper CNN - Training vs Validation Loss', fontweight='bold')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

### 4.6 Compare Training Curves: Baseline vs Deeper

In [ ]:
# Side-by-side training curve comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_base = range(1, len(baseline_history.history['accuracy']) + 1)
epochs_deep = range(1, len(deeper_history.history['accuracy']) + 1)

# Accuracy comparison
axes[0].plot(epochs_base, baseline_history.history['val_accuracy'], 'b--', label='Baseline Val Acc', alpha=0.7)
axes[0].plot(epochs_deep, deeper_history.history['val_accuracy'], 'r-', label='Deeper Val Acc')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Validation Accuracy: Baseline vs Deeper', fontweight='bold')
axes[0].legend()
axes[0].grid(True)

# Loss comparison
axes[1].plot(epochs_base, baseline_history.history['val_loss'], 'b--', label='Baseline Val Loss', alpha=0.7)
axes[1].plot(epochs_deep, deeper_history.history['val_loss'], 'r-', label='Deeper Val Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Validation Loss: Baseline vs Deeper', fontweight='bold')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Print training time comparison
print("Training Time Comparison:")
print(f"  Baseline: {baseline_train_time:.1f}s ({baseline_train_time/60:.1f} min)")
print(f"  Deeper:   {deeper_train_time:.1f}s ({deeper_train_time/60:.1f} min)")
print(f"  Difference: {deeper_train_time - baseline_train_time:.1f}s ({(deeper_train_time/baseline_train_time - 1)*100:.1f}% slower)")

### 4.7 Model Evaluation on Test Set

In [ ]:
# Evaluate deeper model on test set
print('Evaluating deeper model on test set...')
deeper_test_loss, deeper_test_acc = deeper_model.evaluate(test_ds, verbose=1)

print(f'\nDeeper Model - Test Results:')
print(f'  Test Accuracy: {deeper_test_acc:.4f}')
print(f'  Test Loss:     {deeper_test_loss:.4f}')

In [ ]:
# Generate predictions (test_ds is deterministic — no reset needed)
deeper_predictions  = deeper_model.predict(test_ds, verbose=1)
deeper_pred_classes = np.argmax(deeper_predictions, axis=1)
# true_classes was precomputed in cell 41

print(f'Predictions generated: {len(deeper_pred_classes)} samples')

In [ ]:
# Classification report
print("Deeper CNN - Classification Report:")
print("=" * 60)
print(classification_report(true_classes, deeper_pred_classes, target_names=classes))

In [ ]:
# Confusion matrix
cm_deep = confusion_matrix(true_classes, deeper_pred_classes)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Baseline confusion matrix
cm_base = confusion_matrix(true_classes, baseline_pred_classes)
sns.heatmap(cm_base, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes, ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
axes[0].set_title('Baseline CNN', fontweight='bold')

# Deeper confusion matrix
sns.heatmap(cm_deep, annot=True, fmt='d', cmap='Oranges', xticklabels=classes, yticklabels=classes, ax=axes[1])
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')
axes[1].set_title('Deeper CNN', fontweight='bold')

fig.suptitle('Confusion Matrix Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Per-class accuracy comparison
print("Per-Class Accuracy Comparison:")
print("-" * 55)
print(f"{'Class':<12} {'Baseline':<12} {'Deeper':<12} {'Change':<12}")
print("-" * 55)
for i, cls in enumerate(classes):
    cls_mask = true_classes == i
    base_acc = np.mean(baseline_pred_classes[cls_mask] == i)
    deep_acc = np.mean(deeper_pred_classes[cls_mask] == i)
    change = deep_acc - base_acc
    arrow = '↑' if change > 0 else '↓' if change < 0 else '='
    print(f"  {cls:<10} {base_acc:.4f}       {deep_acc:.4f}       {arrow} {abs(change):.4f}")
print("-" * 55)
print(f"  {'OVERALL':<10} {baseline_test_acc:.4f}       {deeper_test_acc:.4f}       {'↑' if deeper_test_acc > baseline_test_acc else '↓'} {abs(deeper_test_acc - baseline_test_acc):.4f}")

### 4.8 Inference on Sample Images

In [ ]:
# Inference on sample test images
fig, axes = plt.subplots(2, 5, figsize=(16, 7))

test_batch, test_labels = next(iter(test_ds))
test_batch  = test_batch.numpy()
test_labels = test_labels.numpy()

for i in range(10):
    row, col   = i // 5, i % 5
    img        = test_batch[i, :, :, 0]
    true_label = classes[np.argmax(test_labels[i])]

    pred_probs = deeper_model.predict(test_batch[i:i+1], verbose=0)
    pred_label = classes[np.argmax(pred_probs)]
    confidence = np.max(pred_probs) * 100

    axes[row, col].imshow(img, cmap='gray')
    color = 'green' if pred_label == true_label else 'red'
    axes[row, col].set_title(f'True: {true_label}\nPred: {pred_label} ({confidence:.1f}%)',
                              fontsize=9, color=color, fontweight='bold')
    axes[row, col].axis('off')

fig.suptitle('Deeper CNN - Sample Predictions (Green=Correct, Red=Wrong)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.9 Deeper Model Observations

**Key comparisons with baseline:**
- Impact of BatchNormalization and Dropout on overfitting
- Effect of doubling filters and layers on accuracy
- Training time trade-off vs performance gain
- Per-class improvements (especially for underrepresented classes)

## 5. Experimentation and Comparative Analysis

This section covers:
1. **Baseline vs Deeper Model Performance** comparison
2. **Computational Efficiency** analysis
3. **Optimizer Analysis**: SGD vs Adam
4. **Ablation Study**: removing Dropout and BatchNormalization
5. **Challenges and Observations**

### 5.1 Baseline vs Deeper Model Performance

In [ ]:
# Comprehensive comparison table
from sklearn.metrics import precision_score, recall_score, f1_score

# Compute metrics for both models
base_precision = precision_score(true_classes, baseline_pred_classes, average='weighted')
base_recall = recall_score(true_classes, baseline_pred_classes, average='weighted')
base_f1 = f1_score(true_classes, baseline_pred_classes, average='weighted')

deep_precision = precision_score(true_classes, deeper_pred_classes, average='weighted')
deep_recall = recall_score(true_classes, deeper_pred_classes, average='weighted')
deep_f1 = f1_score(true_classes, deeper_pred_classes, average='weighted')

print("=" * 60)
print("BASELINE vs DEEPER MODEL - FULL COMPARISON")
print("=" * 60)
print(f"{'Metric':<25} {'Baseline':<15} {'Deeper':<15}")
print("-" * 60)
print(f"{'Test Accuracy':<25} {baseline_test_acc:.4f}         {deeper_test_acc:.4f}")
print(f"{'Test Loss':<25} {baseline_test_loss:.4f}         {deeper_test_loss:.4f}")
print(f"{'Precision (weighted)':<25} {base_precision:.4f}         {deep_precision:.4f}")
print(f"{'Recall (weighted)':<25} {base_recall:.4f}         {deep_recall:.4f}")
print(f"{'F1-Score (weighted)':<25} {base_f1:.4f}         {deep_f1:.4f}")
print(f"{'Training Time':<25} {baseline_train_time:.1f}s          {deeper_train_time:.1f}s")
print(f"{'Parameters':<25} {baseline_model.count_params():,}       {deeper_model.count_params():,}")
print(f"{'Epochs Trained':<25} {len(baseline_history.history["accuracy"])}              {len(deeper_history.history["accuracy"])}")
print("-" * 60)

In [ ]:
# Bar chart comparison of all metrics
fig, ax = plt.subplots(figsize=(10, 5))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
baseline_vals = [baseline_test_acc, base_precision, base_recall, base_f1]
deeper_vals = [deeper_test_acc, deep_precision, deep_recall, deep_f1]

x = np.arange(len(metrics))
width = 0.3

bars1 = ax.bar(x - width/2, baseline_vals, width, label='Baseline CNN', color='#3498db', edgecolor='black')
bars2 = ax.bar(x + width/2, deeper_vals, width, label='Deeper CNN', color='#e74c3c', edgecolor='black')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005, 
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005, 
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylabel('Score')
ax.set_title('Baseline vs Deeper CNN - Performance Metrics', fontweight='bold', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 5.2 Computational Efficiency

In [ ]:
# Computational cost comparison
print("=" * 60)
print("COMPUTATIONAL EFFICIENCY COMPARISON")
print("=" * 60)
print(f"{'Metric':<30} {'Baseline':<15} {'Deeper':<15}")
print("-" * 60)
print(f"{'Total Parameters':<30} {baseline_model.count_params():,}     {deeper_model.count_params():,}")
print(f"{'Training Time':<30} {baseline_train_time:.1f}s          {deeper_train_time:.1f}s")
print(f"{'Time per Epoch':<30} {baseline_train_time/len(baseline_history.history['accuracy']):.1f}s          {deeper_train_time/len(deeper_history.history['accuracy']):.1f}s")
print(f"{'Epochs Trained':<30} {len(baseline_history.history['accuracy'])}              {len(deeper_history.history['accuracy'])}")
print(f"{'Accuracy Gain':<30} {'—':<15} {(deeper_test_acc - baseline_test_acc)*100:+.2f}%")
print("-" * 60)

param_increase = (deeper_model.count_params() / baseline_model.count_params() - 1) * 100
time_increase = (deeper_train_time / baseline_train_time - 1) * 100
acc_increase = (deeper_test_acc - baseline_test_acc) * 100

print(f"\nThe deeper model has {param_increase:.0f}% more parameters")
print(f"and took {time_increase:.0f}% longer to train")
print(f"for a {acc_increase:+.2f}% change in accuracy.")

In [ ]:
# Training time visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart - total time
models = ['Baseline', 'Deeper']
times = [baseline_train_time, deeper_train_time]
colors_bar = ['#3498db', '#e74c3c']
axes[0].bar(models, times, color=colors_bar, edgecolor='black')
for i, t in enumerate(times):
    axes[0].text(i, t + 1, f'{t:.1f}s', ha='center', fontweight='bold')
axes[0].set_ylabel('Time (seconds)')
axes[0].set_title('Total Training Time', fontweight='bold')

# Bar chart - time per epoch
time_per_epoch = [
    baseline_train_time / len(baseline_history.history['accuracy']),
    deeper_train_time / len(deeper_history.history['accuracy'])
]
axes[1].bar(models, time_per_epoch, color=colors_bar, edgecolor='black')
for i, t in enumerate(time_per_epoch):
    axes[1].text(i, t + 0.1, f'{t:.1f}s', ha='center', fontweight='bold')
axes[1].set_ylabel('Time (seconds)')
axes[1].set_title('Time per Epoch', fontweight='bold')

plt.tight_layout()
plt.show()

### 5.3 Optimizer Analysis: SGD vs Adam

In [ ]:
# Build fresh deeper model for SGD training
with strategy.scope():
    deeper_sgd_model = build_deeper_cnn(INPUT_SHAPE, NUM_CLASSES)

    deeper_sgd_model.compile(
        optimizer=SGD(learning_rate=0.01, momentum=0.9),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

print("Deeper model (SGD) compiled inside strategy scope.")
print(f"  Optimizer: SGD (lr=0.01, momentum=0.9)")

In [ ]:
# Train with SGD
early_stopping_sgd = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr_sgd = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

print("Training deeper model with SGD optimizer...")
print()

start_time_sgd = time.time()

sgd_history = deeper_sgd_model.fit(
    train_ds,
    epochs=100,
    validation_data=val_ds,
    callbacks=[early_stopping_sgd, reduce_lr_sgd],
    class_weight=class_weights,
    verbose=1
)

sgd_train_time = time.time() - start_time_sgd
print(f"\nSGD training completed in {sgd_train_time:.1f} seconds ({sgd_train_time/60:.1f} minutes)")

In [ ]:
# Evaluate SGD model
sgd_test_loss, sgd_test_acc = deeper_sgd_model.evaluate(test_ds, verbose=1)
print(f"\nSGD Model - Test Accuracy: {sgd_test_acc:.4f}, Test Loss: {sgd_test_loss:.4f}")

In [ ]:
# Compare Adam vs SGD convergence
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_adam = range(1, len(deeper_history.history['accuracy']) + 1)
epochs_sgd = range(1, len(sgd_history.history['accuracy']) + 1)

# Accuracy
axes[0].plot(epochs_adam, deeper_history.history['val_accuracy'], 'b-', label='Adam', linewidth=2)
axes[0].plot(epochs_sgd, sgd_history.history['val_accuracy'], 'r-', label='SGD', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Validation Accuracy')
axes[0].set_title('Optimizer Comparison - Validation Accuracy', fontweight='bold')
axes[0].legend()
axes[0].grid(True)

# Loss
axes[1].plot(epochs_adam, deeper_history.history['val_loss'], 'b-', label='Adam', linewidth=2)
axes[1].plot(epochs_sgd, sgd_history.history['val_loss'], 'r-', label='SGD', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation Loss')
axes[1].set_title('Optimizer Comparison - Validation Loss', fontweight='bold')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Optimizer comparison summary
print("=" * 60)
print("OPTIMIZER COMPARISON: ADAM vs SGD")
print("=" * 60)
print(f"{'Metric':<25} {'Adam':<15} {'SGD':<15}")
print("-" * 60)
print(f"{'Test Accuracy':<25} {deeper_test_acc:.4f}         {sgd_test_acc:.4f}")
print(f"{'Test Loss':<25} {deeper_test_loss:.4f}         {sgd_test_loss:.4f}")
print(f"{'Training Time':<25} {deeper_train_time:.1f}s          {sgd_train_time:.1f}s")
print(f"{'Epochs Trained':<25} {len(deeper_history.history['accuracy'])}              {len(sgd_history.history['accuracy'])}")
print(f"{'Best Val Accuracy':<25} {max(deeper_history.history['val_accuracy']):.4f}         {max(sgd_history.history['val_accuracy']):.4f}")
print("-" * 60)

winner = 'Adam' if deeper_test_acc > sgd_test_acc else 'SGD'
print(f"\nBetter optimizer for this task: {winner}")

### 5.4 Ablation Study

The ablation study removes one regularization component at a time to measure its individual contribution:
1. **Without Dropout** — keep BatchNorm, remove all Dropout layers
2. **Without BatchNormalization** — keep Dropout, remove all BatchNorm layers

In [ ]:
# Ablation 1: Deeper model WITHOUT Dropout
from tensorflow.keras.regularizers import l2

def build_deeper_no_dropout(input_shape=(48, 48, 1), num_classes=7):
    reg = l2(1e-4)
    inp = tf.keras.Input(shape=input_shape)
    
    x = layers.Conv2D(64, (3, 3), padding='same', kernel_regularizer=reg,
                      activation='relu', kernel_initializer='he_normal')(inp)
    x = layers.Conv2D(64, (3, 3), padding='same', kernel_regularizer=reg,
                      activation='relu', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Conv2D(128, (3, 3), padding='same', kernel_regularizer=reg,
                      activation='relu', kernel_initializer='he_normal')(x)
    x = layers.Conv2D(128, (3, 3), padding='same', kernel_regularizer=reg,
                      activation='relu', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Conv2D(256, (3, 3), padding='same', kernel_regularizer=reg,
                      activation='relu', kernel_initializer='he_normal')(x)
    x = layers.Conv2D(256, (3, 3), padding='same', kernel_regularizer=reg,
                      activation='relu', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Flatten()(x)
    x = layers.Dense(512, kernel_regularizer=reg, activation='relu',
                     kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, kernel_regularizer=reg, activation='relu',
                     kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(128, kernel_regularizer=reg, activation='relu',
                     kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    
    out = layers.Dense(num_classes, activation='softmax')(x)
    return tf.keras.Model(inp, out, name='ablation_no_dropout')

with strategy.scope():
    ablation_no_dropout = build_deeper_no_dropout(INPUT_SHAPE, NUM_CLASSES)
    ablation_no_dropout.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
print(f"Ablation model (No Dropout) built: {ablation_no_dropout.count_params():,} params")

In [ ]:
# Train without Dropout
print("Training ablation model: No Dropout...")
es_abl1 = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
lr_abl1 = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1)

start_abl1 = time.time()
abl1_history = ablation_no_dropout.fit(
    train_ds, epochs=100, validation_data=val_ds,
    callbacks=[es_abl1, lr_abl1], class_weight=class_weights, verbose=1
)
abl1_time = time.time() - start_abl1

abl1_loss, abl1_acc = ablation_no_dropout.evaluate(test_ds, verbose=0)
print(f"\nNo Dropout - Test Accuracy: {abl1_acc:.4f}, Time: {abl1_time:.1f}s")

In [ ]:
# Ablation 2: Deeper model WITHOUT BatchNormalization
from tensorflow.keras.regularizers import l2

def build_deeper_no_batchnorm(input_shape=(48, 48, 1), num_classes=7):
    reg = l2(1e-4)
    inp = tf.keras.Input(shape=input_shape)
    
    x = layers.Conv2D(64, (3, 3), padding='same', kernel_regularizer=reg,
                      activation='relu', kernel_initializer='he_normal')(inp)
    x = layers.Conv2D(64, (3, 3), padding='same', kernel_regularizer=reg,
                      activation='relu', kernel_initializer='he_normal')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Conv2D(128, (3, 3), padding='same', kernel_regularizer=reg,
                      activation='relu', kernel_initializer='he_normal')(x)
    x = layers.Conv2D(128, (3, 3), padding='same', kernel_regularizer=reg,
                      activation='relu', kernel_initializer='he_normal')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Conv2D(256, (3, 3), padding='same', kernel_regularizer=reg,
                      activation='relu', kernel_initializer='he_normal')(x)
    x = layers.Conv2D(256, (3, 3), padding='same', kernel_regularizer=reg,
                      activation='relu', kernel_initializer='he_normal')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Flatten()(x)
    x = layers.Dense(512, kernel_regularizer=reg, activation='relu',
                     kernel_initializer='he_normal')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, kernel_regularizer=reg, activation='relu',
                     kernel_initializer='he_normal')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, kernel_regularizer=reg, activation='relu',
                     kernel_initializer='he_normal')(x)
    x = layers.Dropout(0.3)(x)
    
    out = layers.Dense(num_classes, activation='softmax')(x)
    return tf.keras.Model(inp, out, name='ablation_no_batchnorm')

with strategy.scope():
    ablation_no_bn = build_deeper_no_batchnorm(INPUT_SHAPE, NUM_CLASSES)
    ablation_no_bn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
print(f"Ablation model (No BatchNorm) built: {ablation_no_bn.count_params():,} params")

In [ ]:
# Train without BatchNormalization
print("Training ablation model: No BatchNormalization...")
es_abl2 = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
lr_abl2 = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1)

start_abl2 = time.time()
abl2_history = ablation_no_bn.fit(
    train_ds, epochs=100, validation_data=val_ds,
    callbacks=[es_abl2, lr_abl2], class_weight=class_weights, verbose=1
)
abl2_time = time.time() - start_abl2

abl2_loss, abl2_acc = ablation_no_bn.evaluate(test_ds, verbose=0)
print(f"\nNo BatchNorm - Test Accuracy: {abl2_acc:.4f}, Time: {abl2_time:.1f}s")

### 5.4.1 Ablation Study Results

In [ ]:
# Ablation study results table
print("=" * 70)
print("ABLATION STUDY RESULTS")
print("=" * 70)
print(f"{'Model Variant':<30} {'Test Acc':<12} {'Test Loss':<12} {'Time':<10}")
print("-" * 70)
print(f"{'Deeper (Full: BN + Dropout)':<30} {deeper_test_acc:.4f}       {deeper_test_loss:.4f}       {deeper_train_time:.1f}s")
print(f"{'Without Dropout':<30} {abl1_acc:.4f}       {abl1_loss:.4f}       {abl1_time:.1f}s")
print(f"{'Without BatchNormalization':<30} {abl2_acc:.4f}       {abl2_loss:.4f}       {abl2_time:.1f}s")
print("-" * 70)

# Impact analysis
dropout_impact = deeper_test_acc - abl1_acc
bn_impact = deeper_test_acc - abl2_acc
print(f"\nDropout contribution:           {dropout_impact:+.4f} accuracy")
print(f"BatchNormalization contribution: {bn_impact:+.4f} accuracy")
more_important = 'Dropout' if abs(dropout_impact) > abs(bn_impact) else 'BatchNormalization'
print(f"\nMore impactful regularization: {more_important}")

In [ ]:
# Ablation visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
models_abl = ['Full Model (BN + Dropout)', 'No Dropout (BN only)', 'No BatchNorm (Dropout only)']
accs_abl = [deeper_test_acc, abl1_acc, abl2_acc]
colors_abl = ['#2ecc71', '#e74c3c', '#f39c12']

bars = axes[0].bar(models_abl, accs_abl, color=colors_abl, edgecolor='black')
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005, 
                 f'{bar.get_height():.4f}', ha='center', fontweight='bold')
axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('Ablation Study - Test Accuracy', fontweight='bold')
axes[0].set_ylim(0, max(accs_abl) * 1.15)

# Training curves comparison
ep_full = range(1, len(deeper_history.history['val_accuracy']) + 1)
ep_abl1 = range(1, len(abl1_history.history['val_accuracy']) + 1)
ep_abl2 = range(1, len(abl2_history.history['val_accuracy']) + 1)

axes[1].plot(ep_full, deeper_history.history['val_accuracy'], 'g-', label='Full Model', linewidth=2)
axes[1].plot(ep_abl1, abl1_history.history['val_accuracy'], 'r--', label='No Dropout', linewidth=2)
axes[1].plot(ep_abl2, abl2_history.history['val_accuracy'], color='orange', linestyle='--', label='No BatchNorm', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation Accuracy')
axes[1].set_title('Ablation Study - Training Curves', fontweight='bold')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

### 5.5 Challenges and Observations

In [ ]:
# Summary of all Part A models
print("=" * 75)
print("PART A - COMPLETE MODEL COMPARISON SUMMARY")
print("=" * 75)
print(f"{'Model':<30} {'Accuracy':<12} {'Loss':<12} {'Params':<15} {'Time':<10}")
print("-" * 75)
print(f"{'Baseline CNN':<30} {baseline_test_acc:.4f}       {baseline_test_loss:.4f}       {baseline_model.count_params():,}     {baseline_train_time:.1f}s")
print(f"{'Deeper CNN (Adam)':<30} {deeper_test_acc:.4f}       {deeper_test_loss:.4f}       {deeper_model.count_params():,}   {deeper_train_time:.1f}s")
print(f"{'Deeper CNN (SGD)':<30} {sgd_test_acc:.4f}       {sgd_test_loss:.4f}       {deeper_sgd_model.count_params():,}   {sgd_train_time:.1f}s")
print(f"{'Ablation: No Dropout':<30} {abl1_acc:.4f}       {abl1_loss:.4f}       {ablation_no_dropout.count_params():,}   {abl1_time:.1f}s")
print(f"{'Ablation: No BatchNorm':<30} {abl2_acc:.4f}       {abl2_loss:.4f}       {ablation_no_bn.count_params():,}   {abl2_time:.1f}s")
print("-" * 75)

best_acc = max(baseline_test_acc, deeper_test_acc, sgd_test_acc, abl1_acc, abl2_acc)
best_model_name = ['Baseline', 'Deeper(Adam)', 'Deeper(SGD)', 'No Dropout', 'No BatchNorm'][
    [baseline_test_acc, deeper_test_acc, sgd_test_acc, abl1_acc, abl2_acc].index(best_acc)]
print(f"\nBest performing model in Part A: {best_model_name} ({best_acc:.4f})")

In [ ]:
# Final comparison bar chart
fig, ax = plt.subplots(figsize=(10, 5))

model_names = ['Baseline', 'Deeper\n(Adam)', 'Deeper\n(SGD)', 'No Dropout', 'No BatchNorm']
model_accs = [baseline_test_acc, deeper_test_acc, sgd_test_acc, abl1_acc, abl2_acc]
colors_final = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']

bars = ax.bar(model_names, model_accs, color=colors_final, edgecolor='black')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005, 
            f'{bar.get_height():.4f}', ha='center', fontweight='bold', fontsize=10)

ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('Part A - All Models Test Accuracy Comparison', fontsize=14, fontweight='bold')
ax.set_ylim(0, max(model_accs) * 1.15)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 5.6 Part A Summary

**Key Findings:**
- Deeper architecture with regularization generally improves over the baseline
- BatchNormalization stabilizes training and improves convergence
- Dropout helps prevent overfitting, especially in fully connected layers
- Adam optimizer typically converges faster than SGD for this task
- Class imbalance remains a challenge, particularly for 'disgust' class

**Challenges Faced:**
- Overfitting: the gap between training and validation accuracy
- Class imbalance: underrepresented classes (disgust) have lower accuracy
- Computational cost increases significantly with deeper models
- Finding the right balance between model complexity and generalization

## 6. Part B: Fine-Tuning a Pre-Trained Model (Transfer Learning)

Transfer learning leverages pre-trained model weights from ImageNet to boost performance on our facial expression dataset. This is especially useful when:
- Dataset is relatively small
- Training from scratch is computationally expensive
- Pre-trained models have already learned rich feature representations

**Model chosen: MobileNetV2** — lightweight inverted-residual architecture pre-trained on ImageNet (1.4 M images). Delivers strong accuracy with only ~3.4 M parameters and runs efficiently on both T4 and P100 GPUs.

### 6.1 Prepare Data for Transfer Learning

**Key adjustments for MobileNetV2:**
- Expects **RGB** input; FER images are **48×48 grayscale** — TF repeats the channel to produce 3-channel tensors
- Images resized to **96×96** (2× native size, minimises upscaling blur)
- `mn_preprocess_input` maps pixel values **[0, 255] → [-1, 1]** (MobileNetV2 convention)
- Batch size scales linearly with GPU count via `MirroredStrategy` for full GPU utilisation

In [ ]:
# Transfer learning constants — scaled for MobileNetV2 on T4×2 / P100
TL_IMG_SIZE   = (96, 96)                                    # 2× FER native; avoids heavy blur
TL_BATCH_SIZE = 64 * strategy.num_replicas_in_sync          # 64 per GPU
TL_INPUT_SHAPE = (96, 96, 3)
TL_AUTOTUNE   = tf.data.AUTOTUNE

print(f"Transfer Learning Settings:")
print(f"  Model      : MobileNetV2")
print(f"  Image size : {TL_IMG_SIZE}")
print(f"  Batch size : {TL_BATCH_SIZE}  ({64} per GPU × {strategy.num_replicas_in_sync})")
print(f"  Input shape: {TL_INPUT_SHAPE}")

In [ ]:
# Training tf.data pipeline — replaces ImageDataGenerator for full GPU throughput.
# Augmentation runs on raw [0, 255] tensors (geometric ops — range-safe).
# mn_preprocess_input converts [0, 255] → [-1, 1] after augmentation.

tl_aug = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.10),
    layers.RandomTranslation(0.10, 0.10),
], name='tl_aug')

def make_tl_ds(directory, augment=False):
    ds = tf.keras.utils.image_dataset_from_directory(
        str(directory),
        labels='inferred',
        label_mode='categorical',
        color_mode='rgb',
        batch_size=None,
        image_size=TL_IMG_SIZE,
        shuffle=augment,
        seed=42,
        class_names=sorted(os.listdir(str(directory))),
    )
    ds = ds.cache()
    if augment:
        ds = ds.map(lambda x, y: (tl_aug(x, training=True), y), num_parallel_calls=TL_AUTOTUNE)
    ds = ds.map(
        lambda x, y: (mn_preprocess_input(tf.cast(x, tf.float32)), y),
        num_parallel_calls=TL_AUTOTUNE,
    )
    return ds.batch(TL_BATCH_SIZE).prefetch(TL_AUTOTUNE)

tl_train_ds = make_tl_ds(TRAIN_DIR, augment=True)
print(f"Train dataset : {len(tl_train_ds)} batches  (batch={TL_BATCH_SIZE})")

In [ ]:
tl_val_ds = make_tl_ds(VAL_DIR)
print(f"Validation dataset : {len(tl_val_ds)} batches")

In [ ]:
tl_test_ds = make_tl_ds(TEST_DIR)
print(f"Test dataset : {len(tl_test_ds)} batches")

### 6.2 Load and Adapt MobileNetV2 Pre-Trained Model

In [ ]:
# Load MobileNetV2 pre-trained on ImageNet inside strategy.scope() so
# MirroredStrategy owns all variables from creation — prevents the
# colocate_vars_with error when unfreezing layers later.
with strategy.scope():
    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=TL_INPUT_SHAPE,
    )

print(f"MobileNetV2 base model loaded.")
print(f"  Total layers     : {len(base_model.layers)}")
print(f"  Total parameters : {base_model.count_params():,}")

In [ ]:
# Print MobileNetV2 base architecture
base_model.summary()

In [ ]:
# Build transfer learning model: MobileNetV2 base + custom classification head
with strategy.scope():
    base_model.trainable = False    # Phase 1: freeze entire base

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(128, kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = Dropout(0.3)(x)
    # dtype='float32' keeps output in full precision under mixed_float16
    output = Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

    tl_model = Model(inputs=base_model.input, outputs=output)

print("Transfer learning model built (MobileNetV2 + custom head) inside strategy scope.")

In [ ]:
# Print full model summary
tl_model.summary()

In [ ]:
# Show trainable vs frozen layers
trainable_count = sum(1 for layer in tl_model.layers if layer.trainable)
frozen_count = sum(1 for layer in tl_model.layers if not layer.trainable)
trainable_params = sum(tf.keras.backend.count_params(w) for w in tl_model.trainable_weights)
non_trainable_params = sum(tf.keras.backend.count_params(w) for w in tl_model.non_trainable_weights)

print("Layer Status:")
print(f"  Frozen layers (MobileNetV2 base): {frozen_count}")
print(f"  Trainable layers (custom head): {trainable_count}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Non-trainable parameters: {non_trainable_params:,}")
print(f"  Total parameters: {tl_model.count_params():,}")

### 6.3 Phase 1: Feature Extraction (Frozen Base)

In [ ]:
# Compile for feature extraction phase
with strategy.scope():
    tl_model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

print("Model compiled for feature extraction phase (inside strategy scope).")
print(f"  Optimizer: Adam (lr=0.001)")
print(f"  Only training custom head layers.")

In [ ]:
# Train Phase 1: Feature extraction
EPOCHS_FE = 30

early_stopping_fe = EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True,
    verbose=1
)

reduce_lr_fe = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=4,
    min_lr=1e-6,
    verbose=1
)

print(f"Phase 1: Feature Extraction - Training for up to {EPOCHS_FE} epochs...")
print(f"  MobileNetV2 base : FROZEN")
print(f"  Training only    : Dense + BatchNorm + Dropout layers")
print()

start_time_fe = time.time()

fe_history = tl_model.fit(
    tl_train_ds,
    epochs=EPOCHS_FE,
    validation_data=tl_val_ds,
    callbacks=[early_stopping_fe, reduce_lr_fe],
    class_weight=class_weights,
    verbose=1
)

fe_train_time = time.time() - start_time_fe
print(f"\nPhase 1 completed in {fe_train_time:.1f} seconds ({fe_train_time/60:.1f} minutes)")

In [ ]:
# Phase 1 training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_fe = range(1, len(fe_history.history['accuracy']) + 1)

axes[0].plot(epochs_fe, fe_history.history['accuracy'], 'b-', label='Training Accuracy')
axes[0].plot(epochs_fe, fe_history.history['val_accuracy'], 'r-', label='Validation Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Phase 1 (Feature Extraction) - Accuracy', fontweight='bold')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs_fe, fe_history.history['loss'], 'b-', label='Training Loss')
axes[1].plot(epochs_fe, fe_history.history['val_loss'], 'r-', label='Validation Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Phase 1 (Feature Extraction) - Loss', fontweight='bold')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"Phase 1 Best Validation Accuracy: {max(fe_history.history['val_accuracy']):.4f}")

### 6.4 Phase 2: Fine-Tuning (Unfreeze Top Layers)

In [ ]:
# Unfreeze the last 50 layers of MobileNetV2 (block_13 onwards).
# BN layers stay frozen — preserves ImageNet statistics, avoids instability
# from updating BN with small FER class distributions.
with strategy.scope():
    base_model.trainable = True
    for layer in base_model.layers[:-50]:
        layer.trainable = False
    for layer in base_model.layers[-50:]:
        layer.trainable = not isinstance(layer, BatchNormalization)

unfrozen_n = sum(1 for l in base_model.layers[-50:] if l.trainable)
print(f"Unfrozen conv/dense base layers : {unfrozen_n}")

print("\nLayer trainability after unfreezing:")
print("-" * 50)
for layer in tl_model.layers:
    if hasattr(layer, 'trainable'):
        status = "TRAINABLE" if layer.trainable else "FROZEN"
        print(f"  {layer.name:<35} {status}")

In [ ]:
# Recompile with lower learning rate for fine-tuning
with strategy.scope():
    tl_model.compile(
        optimizer=Adam(learning_rate=0.00005),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

trainable_params_ft = sum(tf.keras.backend.count_params(w) for w in tl_model.trainable_weights)
print(f"Model recompiled for fine-tuning (inside strategy scope).")
print(f"  Optimizer: Adam (lr=0.00005) — lower rate prevents catastrophic forgetting of block5")
print(f"  Trainable parameters now: {trainable_params_ft:,}")

In [ ]:
# Train Phase 2: Fine-tuning
EPOCHS_FT = 20

early_stopping_ft = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr_ft = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

print(f"Phase 2: Fine-Tuning - Training for up to {EPOCHS_FT} epochs...")
print(f"  MobileNetV2 last 50 layers : UNFROZEN  (BN frozen)")
print(f"  Learning rate              : 0.00005  (100x lower than Phase 1)")
print()

start_time_ft = time.time()

ft_history = tl_model.fit(
    tl_train_ds,
    epochs=EPOCHS_FT,
    validation_data=tl_val_ds,
    callbacks=[early_stopping_ft, reduce_lr_ft],
    class_weight=class_weights,
    verbose=1
)

ft_train_time = time.time() - start_time_ft
tl_total_time = fe_train_time + ft_train_time
print(f"\nPhase 2 completed in {ft_train_time:.1f} seconds ({ft_train_time/60:.1f} minutes)")
print(f"Total transfer learning time: {tl_total_time:.1f} seconds ({tl_total_time/60:.1f} minutes)")

In [ ]:
# Phase 2 training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_ft = range(1, len(ft_history.history['accuracy']) + 1)

axes[0].plot(epochs_ft, ft_history.history['accuracy'], 'b-', label='Training Accuracy')
axes[0].plot(epochs_ft, ft_history.history['val_accuracy'], 'r-', label='Validation Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Phase 2 (Fine-Tuning) - Accuracy', fontweight='bold')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs_ft, ft_history.history['loss'], 'b-', label='Training Loss')
axes[1].plot(epochs_ft, ft_history.history['val_loss'], 'r-', label='Validation Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Phase 2 (Fine-Tuning) - Loss', fontweight='bold')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"Phase 2 Best Validation Accuracy: {max(ft_history.history['val_accuracy']):.4f}")

### 6.5 Model Evaluation and Prediction

In [ ]:
# Evaluate fine-tuned model on test set
tl_test_loss, tl_test_acc = tl_model.evaluate(tl_test_ds, verbose=1)

print(f"\nTransfer Learning (MobileNetV2) - Test Results:")
print(f"  Test Accuracy: {tl_test_acc:.4f}")
print(f"  Test Loss:     {tl_test_loss:.4f}")

In [ ]:
# Generate predictions from tf.data test set
tl_predictions  = tl_model.predict(tl_test_ds, verbose=1)
tl_pred_classes = np.argmax(tl_predictions, axis=1)
tl_true_classes = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in tl_test_ds])

print(f"Predictions generated: {len(tl_pred_classes)} samples")

In [ ]:
# Classification report
print("Transfer Learning (MobileNetV2) - Classification Report:")
print("=" * 60)
print(classification_report(tl_true_classes, tl_pred_classes, target_names=classes))

In [ ]:
# Confusion matrix
cm_tl = confusion_matrix(tl_true_classes, tl_pred_classes)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm_tl, annot=True, fmt='d', cmap='Greens', xticklabels=classes, yticklabels=classes, ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title('Transfer Learning (MobileNetV2) - Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Per-class accuracy
print("Transfer Learning (MobileNetV2) - Per-Class Accuracy:")
print("-" * 40)
for i, cls in enumerate(classes):
    cls_mask = tl_true_classes == i
    cls_acc  = np.mean(tl_pred_classes[cls_mask] == i)
    print(f"  {cls:<10}: {cls_acc:.4f} ({np.sum(cls_mask)} samples)")

### 6.6 Inference on Sample Images

In [ ]:
# Inference on sample test images
fig, axes = plt.subplots(2, 5, figsize=(16, 7))

test_batch_tl, test_labels_tl = next(iter(tl_test_ds))
test_batch_tl  = test_batch_tl.numpy()
test_labels_tl = test_labels_tl.numpy()

for i in range(min(10, len(test_batch_tl))):
    row, col   = i // 5, i % 5
    # Reverse mn_preprocess_input: [-1,1] → [0,1] for display
    img_display = np.clip((test_batch_tl[i] + 1.0) / 2.0, 0, 1)
    true_label  = classes[np.argmax(test_labels_tl[i])]

    pred_probs  = tl_model.predict(test_batch_tl[i:i+1], verbose=0)
    pred_label  = classes[np.argmax(pred_probs)]
    confidence  = np.max(pred_probs) * 100

    axes[row, col].imshow(img_display)
    color = 'green' if pred_label == true_label else 'red'
    axes[row, col].set_title(
        f"True: {true_label}\nPred: {pred_label} ({confidence:.1f}%)",
        fontsize=9, color=color, fontweight='bold'
    )
    axes[row, col].axis('off')

fig.suptitle('MobileNetV2 Transfer Learning - Sample Predictions (Green=Correct, Red=Wrong)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 6.7 Compare All Three Models: Baseline vs Deeper vs Transfer Learning

In [ ]:
# Compute metrics for transfer learning model
tl_precision = precision_score(tl_true_classes, tl_pred_classes, average='weighted')
tl_recall    = recall_score(tl_true_classes, tl_pred_classes, average='weighted')
tl_f1        = f1_score(tl_true_classes, tl_pred_classes, average='weighted')

print("=" * 78)
print("FINAL COMPARISON: ALL THREE MODELS")
print("=" * 78)
print(f"{'Metric':<25} {'Baseline':<15} {'Deeper':<15} {'MobileNetV2 (TL)':<18}")
print("-" * 78)
print(f"{'Test Accuracy':<25} {baseline_test_acc:.4f}         {deeper_test_acc:.4f}         {tl_test_acc:.4f}")
print(f"{'Test Loss':<25} {baseline_test_loss:.4f}         {deeper_test_loss:.4f}         {tl_test_loss:.4f}")
print(f"{'Precision (weighted)':<25} {base_precision:.4f}         {deep_precision:.4f}         {tl_precision:.4f}")
print(f"{'Recall (weighted)':<25} {base_recall:.4f}         {deep_recall:.4f}         {tl_recall:.4f}")
print(f"{'F1-Score (weighted)':<25} {base_f1:.4f}         {deep_f1:.4f}         {tl_f1:.4f}")
print(f"{'Training Time':<25} {baseline_train_time:.1f}s          {deeper_train_time:.1f}s          {tl_total_time:.1f}s")
print(f"{'Parameters':<25} {baseline_model.count_params():,}     {deeper_model.count_params():,}   {tl_model.count_params():,}")
print("-" * 78)

all_accs  = [baseline_test_acc, deeper_test_acc, tl_test_acc]
best_name = ['Baseline CNN', 'Deeper CNN', 'MobileNetV2 Transfer Learning'][all_accs.index(max(all_accs))]
print(f"\nBest Overall Model: {best_name} ({max(all_accs):.4f} accuracy)")

In [ ]:
# Final bar chart: all three models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics   = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
base_vals = [baseline_test_acc, base_precision, base_recall, base_f1]
deep_vals = [deeper_test_acc,   deep_precision, deep_recall, deep_f1]
tl_vals   = [tl_test_acc,       tl_precision,   tl_recall,   tl_f1]

x     = np.arange(len(metrics))
width = 0.25

axes[0].bar(x - width, base_vals, width, label='Baseline',         color='#3498db', edgecolor='black')
axes[0].bar(x,         deep_vals, width, label='Deeper',           color='#e74c3c', edgecolor='black')
axes[0].bar(x + width, tl_vals,   width, label='MobileNetV2 (TL)', color='#2ecc71', edgecolor='black')
axes[0].set_ylabel('Score')
axes[0].set_title('All Models - Performance Metrics', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].legend()
axes[0].set_ylim(0, 1.1)
axes[0].grid(axis='y', alpha=0.3)

per_class_base = [np.mean(baseline_pred_classes[true_classes == i] == i) for i in range(NUM_CLASSES)]
per_class_deep = [np.mean(deeper_pred_classes[true_classes == i] == i)   for i in range(NUM_CLASSES)]
per_class_tl   = [np.mean(tl_pred_classes[tl_true_classes == i] == i)    for i in range(NUM_CLASSES)]

x2 = np.arange(NUM_CLASSES)
axes[1].bar(x2 - width, per_class_base, width, label='Baseline',         color='#3498db', edgecolor='black')
axes[1].bar(x2,         per_class_deep, width, label='Deeper',           color='#e74c3c', edgecolor='black')
axes[1].bar(x2 + width, per_class_tl,   width, label='MobileNetV2 (TL)', color='#2ecc71', edgecolor='black')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('All Models - Per-Class Accuracy', fontweight='bold')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(classes, rotation=45)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side confusion matrices for all three models
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

cm_base = confusion_matrix(true_classes,    baseline_pred_classes)
cm_deep = confusion_matrix(true_classes,    deeper_pred_classes)
cm_tl   = confusion_matrix(tl_true_classes, tl_pred_classes)

sns.heatmap(cm_base, annot=True, fmt='d', cmap='Blues',  xticklabels=classes, yticklabels=classes, ax=axes[0])
axes[0].set_title('Baseline CNN', fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

sns.heatmap(cm_deep, annot=True, fmt='d', cmap='Oranges', xticklabels=classes, yticklabels=classes, ax=axes[1])
axes[1].set_title('Deeper CNN', fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')

sns.heatmap(cm_tl, annot=True, fmt='d', cmap='Greens', xticklabels=classes, yticklabels=classes, ax=axes[2])
axes[2].set_title('MobileNetV2 (Transfer Learning)', fontweight='bold')
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('True')

fig.suptitle('Confusion Matrix Comparison - All Three Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 6.8 Final Discussion

**Transfer Learning vs Training from Scratch:**
- MobileNetV2 leverages pre-trained ImageNet features that capture low-level patterns (edges, textures) transferable to facial expressions
- Its inverted residual blocks with linear bottlenecks provide strong feature extraction at only ~3.4 M parameters — far more efficient than VGG16 (138 M)
- Feature extraction phase trains quickly since only the custom head is updated
- Fine-tuning the last 50 layers with a 100× lower learning rate allows domain adaptation without catastrophic forgetting of ImageNet features
- Resizing from 48×48 to 96×96 (2× scale) gives the model better spatial context while minimising upscaling distortion

**Key Takeaways:**
- Transfer learning is particularly effective when the dataset is small or class-imbalanced
- MobileNetV2 is significantly more parameter-efficient than VGG16, making it better suited for mobile/deployment scenarios
- Keeping BatchNorm layers frozen during fine-tuning is critical — it preserves ImageNet batch statistics and avoids instability
- `MirroredStrategy` + `mixed_float16` fully utilises both T4 GPUs, halving training time versus single-GPU float32
- Class imbalance (especially 'disgust') remains the biggest challenge across all models